# 25 — OOF Tail Risk and Spatial Support Audit

This notebook creates no feature, correction or submission. It persists leakage-controlled row-level OOF predictions
and diagnoses tail risk and train/test support. Residual is fixed as **target − prediction**; positive means
underprediction.

## Frozen methodology

Primary strategy is two complete-date strict temporal folds. It is called **canonical-equivalent leakage-controlled
OOF**, not exact canonical OOF: the expert/blend structure is retained while the risky location-major
`prior_global_month` is replaced by strict date-grouped history. Large-error thresholds are OOF P90, OOF P95 and
2× median absolute error. Spatial support is `0.50 seen-location + 0.25 exp(-nearest_km/250) + 0.25 min(history/24,1)`;
temporal support is a fixed weighted combination of forecast gap, seasonal history and location recency. Combined
support is their arithmetic mean. No threshold is tuned against results.

## Input gate, cached OOF generation and audit

Fold caches are reused only when their frozen policy key matches. Each validation date is later than its training
maximum, validation targets are never feature inputs, and every primary OOF row is unique.

In [1]:
from pathlib import Path
from time import perf_counter
import hashlib, json, warnings, gc
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, skew, kurtosis
from sklearn.metrics import root_mean_squared_error, mean_absolute_error
from sklearn.neighbors import BallTree
from lightgbm import LGBMRegressor
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
SEED=42; RNG=np.random.default_rng(SEED); T0=perf_counter(); EARTH_KM=6371.0088
ROOT=Path.cwd().resolve()
if not (ROOT/"data/raw/Train.csv").exists(): ROOT=ROOT.parent
DATA=ROOT/"data/raw"; SUB=ROOT/"submissions"; ART=ROOT/"artifacts/notebook25"; ART.mkdir(parents=True,exist_ok=True)
CANON=SUB/"submission_optimized.csv"; BEST=SUB/"submission_21_spatial_residual_candidate.csv"
CANON_SHA="b61ecd15862ca6d7527c4f494de802ed6ea6d31c420737cb78f977d9bda3b968"; BEST_SHA="84d4ce6efdbfc3fbca55d9eea1ebccfe628ffa962e31b62a654c179dc649b488"
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def stamp(s): print(f"[{s}] elapsed={perf_counter()-T0:.1f}s")
train=pd.read_csv(DATA/"Train.csv",parse_dates=["time"]); test=pd.read_csv(DATA/"Test.csv",parse_dates=["time"]); sample=pd.read_csv(DATA/"SampleSubmission.csv")
canon=pd.read_csv(CANON); best=pd.read_csv(BEST)
input_rows=[
("Training data",(DATA/"Train.csv").exists(),len(train),train.sample_id.is_unique,"target" in train,"time" in train,"lat/lon" if {"lat","lon"}<=set(train) else "missing",int(train.isna().sum().sum()),"source order retained"),
("Test data",(DATA/"Test.csv").exists(),len(test),test.ID.is_unique,"TWS_t" in test,"time" in test,"lat/lon" if {"lat","lon"}<=set(test) else "missing",int(test.isna().sum().sum()),"source order retained"),
("Canonical submission",CANON.exists(),len(canon),canon.ID.is_unique,"Target" in canon,False,"ID",int(canon.isna().sum().sum()),sha(CANON)),
("Current-best submission",BEST.exists(),len(best),best.ID.is_unique,"Target" in best,False,"ID",int(best.isna().sum().sum()),sha(BEST)),
("Notebook 20 validation definitions",(ROOT/"artifacts/notebook20/strict_temporal_folds.csv").exists(),2,True,True,True,"global grid",0,"artifact present"),
("Serialized model artifacts",bool(list((ROOT/"models").glob("*"))),len(list((ROOT/"models").glob("*"))),True,False,False,"not applicable",0,"not required; fold models are reproducibly refit")]
integrity=pd.DataFrame(input_rows,columns=["Input","Exists","Row/item count","ID unique","Target column","Date column","Spatial identifier","Missing values","Ordering/hash note"])
integrity["Canonical SHA match"]=sha(CANON)==CANON_SHA; integrity["Current-best SHA match"]=sha(BEST)==BEST_SHA
integrity.to_csv(ART/"input_integrity_report.csv",index=False); assert sha(CANON)==CANON_SHA and sha(BEST)==BEST_SHA and canon.ID.equals(sample.ID)

# Primary strategy is fixed before results. Space-time has inadequate global row coverage.
dates=np.array(sorted(train.time.unique())); specs=[("strict_temporal_1",len(dates)-48,len(dates)-24),("strict_temporal_2",len(dates)-24,len(dates))]
valid_indices=np.concatenate([train.index[train.time.isin(dates[a:b])].to_numpy() for _,a,b in specs])
strategy=pd.DataFrame([{"Strategy name":"canonical-equivalent leakage-controlled strict temporal OOF","Terminology":"canonical-equivalent OOF, not exact canonical OOF",
 "Fold count":2,"Date ordering":"complete date groups; train max < validation min","Spatial separation":"none; global same-grid temporal transfer",
 "Row coverage":len(np.unique(valid_indices))/len(train),"Excluded row count":len(train)-len(np.unique(valid_indices)),
 "Why selected":"Space-time coverage is regional; strict temporal gives the highest leakage-free global row coverage and is reproducible.",
 "Limitations":"Canonical prior_global_month ordering is replaced by strict date-grouped prior construction; therefore this is not exact canonical OOF. Same locations are generally seen."}])
strategy.to_csv(ART/"oof_strategy_definition.csv",index=False)

SPEI=["SPEI_01_t","SPEI_03_t","SPEI_06_t","SPEI_12_t"]
def static(df):
 o=df.copy(); dt=o.time; o["month"]=dt.dt.month.astype("int8"); o["quarter"]=dt.dt.quarter.astype("int8"); o["year"]=dt.dt.year.astype("int16")
 o["elapsed_month"]=((dt.dt.year-train.time.dt.year.min())*12+dt.dt.month-1).astype("int16"); o["month_sin_calc"]=np.sin(2*np.pi*o.month/12); o["month_cos_calc"]=np.cos(2*np.pi*o.month/12)
 o["season_sin"]=np.sin(2*np.pi*o.quarter/4); o["season_cos"]=np.cos(2*np.pi*o.quarter/4); s=o[SPEI]
 o["spei_mean"]=s.mean(1); o["spei_min"]=s.min(1); o["spei_max"]=s.max(1); o["spei_std"]=s.std(1); o["spei_01_minus_12"]=o.SPEI_01_t-o.SPEI_12_t
 o["spei_03_minus_12"]=o.SPEI_03_t-o.SPEI_12_t; o["spei_short_long_ratio"]=o.SPEI_01_t/(o.SPEI_12_t.abs()+.1)
 o["lat_band_5"]=np.floor((o.lat+90)/5); o["lon_band_5"]=np.floor((o.lon+180)/5); o["spatial_bin_10"]=o.lat_band_5*100+o.lon_band_5
 o["lat_lon_interaction"]=o.lat*o.lon/1000; o["lat_sin"]=np.sin(np.radians(o.lat)); o["lat_cos"]=np.cos(np.radians(o.lat)); o["lon_sin"]=np.sin(np.radians(o.lon)); o["lon_cos"]=np.cos(np.radians(o.lon)); return o
X=static(train)
BASE=["lat","lon","TWS_t",*SPEI,"SOIL_MOISTURE_t","month","month_sin","month_cos","year","quarter","elapsed_month","month_sin_calc","month_cos_calc","season_sin","season_cos",
"spei_mean","spei_min","spei_max","spei_std","spei_01_minus_12","spei_03_minus_12","spei_short_long_ratio","lat_band_5","lon_band_5","spatial_bin_10","lat_lon_interaction","lat_sin","lat_cos","lon_sin","lon_cos"]
MASK=[x for x in BASE if x!="TWS_t"]; PRI=["prior_location_mean","prior_location_std","prior_location_month","prior_global_month"]
PARAM=dict(n_estimators=160,learning_rate=.08,num_leaves=63,min_child_samples=100,reg_lambda=1,subsample=.85,colsample_bytree=.9,random_state=SEED,n_jobs=4,verbosity=-1)
def lgb(): return LGBMRegressor(**PARAM)
def strict_global_month(o):
 q=o.groupby(["month","time"],observed=True).target.agg(["sum","count"]).reset_index().sort_values(["month","time"])
 q["cs"]=q.groupby("month")["sum"].cumsum()-q["sum"]; q["cc"]=q.groupby("month")["count"].cumsum()-q["count"]; q["prior_global_month"]=q.cs/q.cc.replace(0,np.nan)
 return o.merge(q[["month","time","prior_global_month"]],on=["month","time"],how="left",sort=False)["prior_global_month"].to_numpy()
def train_prior(df):
 o=df.sort_values(["lat","lon","time"]).copy(); sh=o.groupby(["lat","lon"]).target.shift(1)
 o["prior_location_mean"]=sh.groupby([o.lat,o.lon]).expanding().mean().reset_index(level=[0,1],drop=True); o["prior_location_std"]=sh.groupby([o.lat,o.lon]).expanding().std().reset_index(level=[0,1],drop=True)
 o["prior_location_month"]=sh.groupby([o.lat,o.lon,o.month]).expanding().mean().reset_index(level=[0,1,2],drop=True); o["prior_global_month"]=strict_global_month(o); return o.sort_index()
def apply_prior(h,a):
 loc=h.groupby(["lat","lon"]).target.agg(["mean","std"]).rename(columns={"mean":"prior_location_mean","std":"prior_location_std"}); lm=h.groupby(["lat","lon","month"]).target.agg(["sum","count"]).rename(columns={"sum":"s","count":"n"})
 gm=h.groupby("month").target.mean().rename("prior_global_month"); o=a.join(loc,on=["lat","lon"]).join(lm,on=["lat","lon","month"]).join(gm,on="month"); o["prior_location_month"]=(o.s+4*o.prior_location_mean)/(o.n.fillna(0)+4); return o.drop(columns=["s","n"])
known_positions=set(np.flatnonzero(test.groupby("time").TWS_t.count().to_numpy()>1000))

cache_meta=ART/"oof_cache_manifest.json"; policy_key=hashlib.sha256((CANON_SHA+str(PARAM)+"strict_global_month_v1").encode()).hexdigest()
cached=[]
if cache_meta.exists():
 meta=json.loads(cache_meta.read_text()); cached=meta.get("policy_key")==policy_key and all((ART/f"_cache_{n}.parquet").exists() for n,_,_ in specs)
fold_frames=[]; quality=[]
for name,a,b in specs:
 cache=ART/f"_cache_{name}.parquet"
 tr=X[X.time.isin(dates[:a])]; va=X[X.time.isin(dates[a:b])]; assert tr.time.max()<va.time.min() and set(tr.time.unique()).isdisjoint(va.time.unique())
 if cached: out=pd.read_parquet(cache); cache_used=True
 else:
  vd=np.array(sorted(va.time.unique())); is_known=va.time.isin({d for j,d in enumerate(vd) if j in known_positions}).to_numpy()
  km=lgb().fit(tr[BASE],tr.target); bm=lgb().fit(tr[MASK],tr.target); tp=train_prior(tr); vp=apply_prior(tr,va); hm=lgb().fit(tp[MASK+PRI],tp.target)
  pred=np.empty(len(va)); pred[is_known]=km.predict(va.loc[is_known,BASE]); pred[~is_known]=.65*bm.predict(va.loc[~is_known,MASK])+.35*hm.predict(vp.loc[~is_known,MASK+PRI])
  out=va[["sample_id","time","lat","lon","target","SPEI_03_t","SPEI_12_t","SOIL_MOISTURE_t"]].copy(); out["original_index"]=va.index; out["oof_prediction"]=pred; out["known_simulation"]=is_known; out["fold_id"]=name
  out.to_parquet(cache,index=False); cache_used=False
 fold_frames.append(out)
 quality.append({"Fold":name,"Train row count":len(tr),"Validation row count":len(va),"Train date max":tr.time.max(),"Validation date min":va.time.min(),"Date overlap":len(set(tr.time)&set(va.time)),
  "Spatial overlap":len(set(zip(va.lat,va.lon))&set(zip(tr.lat,tr.lon)))/len(set(zip(va.lat,va.lon))),"OOF RMSE":root_mean_squared_error(out.target,out.oof_prediction),"MAE":mean_absolute_error(out.target,out.oof_prediction),
  "Mean residual":(out.target-out.oof_prediction).mean(),"Prediction minimum":out.oof_prediction.min(),"Prediction maximum":out.oof_prediction.max(),"Target minimum":out.target.min(),"Target maximum":out.target.max(),"Cache used":cache_used})
 stamp(name); del tr,va; gc.collect()
if not cached: cache_meta.write_text(json.dumps({"policy_key":policy_key,"folds":[x[0] for x in specs]},indent=2))
oof=pd.concat(fold_frames,ignore_index=True); assert oof.original_index.is_unique and np.isfinite(oof.oof_prediction).all()
oof["residual"]=oof.target-oof.oof_prediction; oof["absolute_error"]=oof.residual.abs(); oof["squared_error"]=oof.residual**2
oof["underprediction_flag"]=oof.residual>0; oof["overprediction_flag"]=oof.residual<0; assert np.allclose(oof.residual,oof.target-oof.oof_prediction)
pd.DataFrame(quality).to_csv(ART/"oof_quality_report.csv",index=False)

def add_groups(d):
 d=d.copy(); d["year"]=d.time.dt.year.astype("int16"); d["month"]=d.time.dt.month.astype("int8"); d["hemisphere"]=np.where(d.lat<0,"South","North")
 d["latitude_band"]=(np.floor(d.lat/10)*10).astype(int).astype(str); d["spatial_group"]=d.latitude_band+"|"+(np.floor(d.lon/20)*20).astype(int).astype(str)
 d["climate_regime"]=pd.cut(d.SPEI_12_t,[-np.inf,-1,1,np.inf],labels=["dry","normal","wet"]).astype(str)
 d["drought_regime"]=pd.cut(d.SPEI_03_t,[-np.inf,-1.5,-.5,.5,1.5,np.inf],labels=["severe_dry","dry","neutral","wet","very_wet"]).astype(str); return d
oof=add_groups(oof)

def spatial_support(query,history):
 hu=history[["lat","lon"]].drop_duplicates().reset_index(drop=True); qu=query[["lat","lon"]].drop_duplicates().reset_index(drop=True)
 tree=BallTree(np.radians(hu[["lat","lon"]].to_numpy()),metric="haversine"); k=min(6,len(hu)); dist,_=tree.query(np.radians(qu[["lat","lon"]].to_numpy()),k=k); dist*=EARTH_KM
 qu["nearest_training_km"]=dist[:,0]; qu["knn5_mean_km"]=dist[:,1:min(6,k)].mean(1) if k>1 else dist[:,0]
 loc_count=history.groupby(["lat","lon"]).size(); qu["location_history_count"]=[loc_count.get((a,b),0) for a,b in zip(qu.lat,qu.lon)]
 band=history.assign(b=(np.floor(history.lat/10)*10).astype(int)).groupby("b").size(); qu["latitude_band_history_count"]=[band.get(int(np.floor(x/10)*10),0) for x in qu.lat]
 qu["seen_location"]=qu.location_history_count>0; qu["inside_training_bbox_proxy"]=(qu.lat.between(history.lat.min(),history.lat.max())&qu.lon.between(history.lon.min(),history.lon.max()))
 qu["spatial_density"]=qu.location_history_count/(qu.knn5_mean_km+1); qu["spatial_support_score"]=.5*qu.seen_location+.25*np.exp(-qu.nearest_training_km/250)+.25*np.minimum(qu.location_history_count/24,1)
 qu["spatial_support_group"]=pd.cut(qu.spatial_support_score,[-np.inf,.35,.60,.80,np.inf],labels=["Out-of-support","Low support","Medium support","High support"]).astype(str)
 return query[["lat","lon"]].merge(qu,on=["lat","lon"],how="left")
def temporal_support(query,history):
 q=query[["time","lat","lon"]].copy(); end=history.time.max(); q["train_end_date"]=end
 q["gap_months"]=(q.time.dt.year-end.year)*12+q.time.dt.month-end.month; seasonal=history.groupby(history.time.dt.month).time.nunique(); q["same_month_historical_dates"]=q.time.dt.month.map(seasonal).fillna(0)
 last=history.groupby(["lat","lon"]).time.max(); last_dates=pd.Series(list(zip(q.lat,q.lon))).map(last); q["months_since_location_observation"]=[(t.year-l.year)*12+t.month-l.month if pd.notna(l) else 999 for t,l in zip(q.time,last_dates)]
 q["date_range_extrapolation"]=q.time>end; q["temporal_support_score"]=.4*np.exp(-np.maximum(q.gap_months-1,0)/12)+.3*np.minimum(q.same_month_historical_dates/10,1)+.3*np.exp(-q.months_since_location_observation/12)
 q["temporal_support_group"]=pd.cut(q.temporal_support_score,[-np.inf,.35,.60,.80,np.inf],labels=["Temporal extrapolation","Low temporal support","Medium temporal support","High temporal support"]).astype(str); return q

# Fold-specific support avoids using future validation rows as history.
sp_parts=[]; tm_parts=[]
for name,a,b in specs:
 idx=oof.fold_id.eq(name); hist=train[train.time.isin(dates[:a])]; q=oof.loc[idx,["sample_id","time","lat","lon"]]
 sp=spatial_support(q,hist); sp["row_id"]=q.sample_id.to_numpy(); sp["Dataset"]="OOF"; sp_parts.append(sp)
 tm=temporal_support(q,hist); tm["row_id"]=q.sample_id.to_numpy(); tm["Dataset"]="OOF"; tm_parts.append(tm)
test_q=test[["ID","time","lat","lon"]].rename(columns={"ID":"sample_id"}); sp_test=spatial_support(test_q,train); sp_test["row_id"]=test_q.sample_id; sp_test["Dataset"]="test"
tm_test=temporal_support(test_q,train); tm_test["row_id"]=test_q.sample_id; tm_test["Dataset"]="test"
spatial_rows=pd.concat([*sp_parts,sp_test],ignore_index=True); temporal_rows=pd.concat([*tm_parts,tm_test],ignore_index=True)
spatial_rows.to_parquet(ART/"spatial_support_rows.parquet",index=False); spatial_rows.to_csv(ART/"spatial_support_rows.csv",index=False); spatial_rows.to_csv(ART/"spatial_support_rows.csv.gz",index=False,compression="gzip")
temporal_rows.to_parquet(ART/"temporal_support_rows.parquet",index=False); temporal_rows.to_csv(ART/"temporal_support_rows.csv",index=False); temporal_rows.to_csv(ART/"temporal_support_rows.csv.gz",index=False,compression="gzip")
oof=oof.merge(spatial_rows[spatial_rows.Dataset=="OOF"][["row_id","nearest_training_km","knn5_mean_km","location_history_count","latitude_band_history_count","seen_location","inside_training_bbox_proxy","spatial_density","spatial_support_score","spatial_support_group"]],left_on="sample_id",right_on="row_id",how="left").drop(columns="row_id")
oof=oof.merge(temporal_rows[temporal_rows.Dataset=="OOF"][["row_id","train_end_date","gap_months","same_month_historical_dates","months_since_location_observation","date_range_extrapolation","temporal_support_score","temporal_support_group"]],left_on="sample_id",right_on="row_id",how="left").drop(columns="row_id")
oof["combined_support_score"]=(oof.spatial_support_score+oof.temporal_support_score)/2; oof["row_id"]=oof.sample_id; oof["date"]=oof.time; oof["latitude"]=oof.lat; oof["longitude"]=oof.lon
cols=["row_id","original_index","date","year","month","latitude","longitude","spatial_group","hemisphere","latitude_band","climate_regime","drought_regime","fold_id","train_end_date","target","oof_prediction","residual","absolute_error","squared_error","underprediction_flag","overprediction_flag","spatial_support_score","temporal_support_score","combined_support_score"]
oof[cols].to_parquet(ART/"oof_row_level_predictions.parquet",index=False); oof[cols].to_csv(ART/"oof_row_level_predictions.csv.gz",index=False,compression="gzip")
stamp("row-level OOF and support")

P90=oof.absolute_error.quantile(.90); P95=oof.absolute_error.quantile(.95); TWO_MED=2*oof.absolute_error.median()
metrics={"RMSE":np.sqrt(oof.squared_error.mean()),"MAE":oof.absolute_error.mean(),"Mean residual":oof.residual.mean(),"Median residual":oof.residual.median(),"Residual standard deviation":oof.residual.std(),
 "Median absolute error":oof.absolute_error.median(),"P75 absolute error":oof.absolute_error.quantile(.75),"P90 absolute error":P90,"P95 absolute error":P95,"P99 absolute error":oof.absolute_error.quantile(.99),
 "Maximum absolute error":oof.absolute_error.max(),"Residual skewness":skew(oof.residual),"Residual kurtosis":kurtosis(oof.residual),"Underprediction rate":oof.underprediction_flag.mean(),"Overprediction rate":oof.overprediction_flag.mean(),
 "Large-error fraction >P90":(oof.absolute_error>P90).mean(),"Large-error fraction >P95":(oof.absolute_error>P95).mean(),"Large-error fraction >2x median":(oof.absolute_error>TWO_MED).mean()}
pd.DataFrame([{"Metric":k,"Value":v,"Threshold definition":"P90/P95 are OOF-derived; 2x median fixed from OOF"} for k,v in metrics.items()]).to_csv(ART/"global_tail_risk_metrics.csv",index=False)

def reason(r):
 tags=[]
 if r.spatial_support_group in ["Low support","Out-of-support"]:tags.append("Low spatial support")
 if r.temporal_support_group in ["Low temporal support","Temporal extrapolation"]:tags.append("Temporal extrapolation")
 if abs(r.target)>=oof.target.abs().quantile(.95):tags.append("Extreme target")
 if (oof.climate_regime==r.climate_regime).mean()<.1:tags.append("Climate-regime rarity")
 tags.append("Model underprediction" if r.residual>0 else "Model overprediction"); return "; ".join(tags) if tags else "Unknown"
worst=oof.nlargest(100,"absolute_error").copy(); worst["Nearby training density"]=worst.spatial_density; worst["Possible reason category"]=worst.apply(reason,axis=1)
worst[["date","latitude","longitude","spatial_group","target","oof_prediction","residual","absolute_error","fold_id","climate_regime","drought_regime","spatial_support_score","temporal_support_score","Nearby training density","Possible reason category"]].to_csv(ART/"worst_error_rows.csv",index=False)

def perf_group(d,col):
 rows=[]
 for key,g in d.groupby(col,observed=True): rows.append({col:key,"Row count":len(g),"RMSE":np.sqrt(g.squared_error.mean()),"MAE":g.absolute_error.mean(),"P95 absolute error":g.absolute_error.quantile(.95),"Mean residual":g.residual.mean(),"Large-error fraction":(g.absolute_error>P95).mean()})
 return pd.DataFrame(rows)
perf_group(oof,"spatial_support_group").to_csv(ART/"spatial_support_performance.csv",index=False); perf_group(oof,"temporal_support_group").to_csv(ART/"temporal_support_performance.csv",index=False)

def dist_rows(label,d):
 return {"Dataset":label,"Rows":len(d),"Nearest mean km":d.nearest_training_km.mean(),"Nearest p50 km":d.nearest_training_km.median(),"Nearest p95 km":d.nearest_training_km.quantile(.95),"Density mean":d.spatial_density.mean(),
 "Out-of-support fraction":d.spatial_support_group.eq("Out-of-support").mean(),"Low-or-out fraction":d.spatial_support_group.isin(["Low support","Out-of-support"]).mean(),"Seen-location fraction":d.seen_location.mean(),"Unseen-location fraction":1-d.seen_location.mean(),
 "Southern fraction":(d.lat<0).mean(),"Latitude bands":d.assign(b=(np.floor(d.lat/10)*10).astype(int)).b.value_counts().to_json(),"Low-representation regions":"latitude bands below 1% are listed in distribution JSON"}
pd.DataFrame([dist_rows("OOF",spatial_rows[spatial_rows.Dataset=="OOF"]),dist_rows("test",sp_test)]).to_csv(ART/"test_spatial_support_audit.csv",index=False)

rho_abs=spearmanr(oof.combined_support_score,oof.absolute_error).statistic; rho_sq=spearmanr(oof.combined_support_score,oof.squared_error).statistic
boot=[]
for _ in range(120):
 z=oof.sample(min(20000,len(oof)),replace=True,random_state=int(RNG.integers(1e9))); boot.append(spearmanr(z.combined_support_score,z.absolute_error).statistic)
oof["support_quartile"]=pd.qcut(oof.combined_support_score,4,labels=["Q1 low","Q2","Q3","Q4 high"],duplicates="drop")
combo=perf_group(oof,"support_quartile"); combo["Spearman support vs absolute error"]=rho_abs; combo["Spearman support vs squared error"]=rho_sq; combo["Bootstrap CI low"]=np.quantile(boot,.025); combo["Bootstrap CI high"]=np.quantile(boot,.975)
patterns={"low spatial + low temporal":oof.spatial_support_score.lt(.6)&oof.temporal_support_score.lt(.6),"high spatial + low temporal":oof.spatial_support_score.ge(.8)&oof.temporal_support_score.lt(.6),"low spatial + high temporal":oof.spatial_support_score.lt(.6)&oof.temporal_support_score.ge(.8),"high spatial + high temporal":oof.spatial_support_score.ge(.8)&oof.temporal_support_score.ge(.8)}
for k,m in patterns.items():
 g=oof[m]; combo=pd.concat([combo,pd.DataFrame([{"support_quartile":k,"Row count":len(g),"RMSE":np.sqrt(g.squared_error.mean()) if len(g) else np.nan,"MAE":g.absolute_error.mean(),"P95 absolute error":g.absolute_error.quantile(.95),"Mean residual":g.residual.mean(),"Large-error fraction":(g.absolute_error>P95).mean(),"Spearman support vs absolute error":rho_abs,"Spearman support vs squared error":rho_sq,"Bootstrap CI low":np.quantile(boot,.025),"Bootstrap CI high":np.quantile(boot,.975)}])],ignore_index=True)
combo.to_csv(ART/"combined_support_error_relationship.csv",index=False)

def tail_table(defs,type_name):
 rows=[]
 for name,m in defs.items():
  g=oof[m]
  if not len(g):continue
  rows.append({"Group type":type_name,"Group":name,"Row count":len(g),"RMSE":np.sqrt(g.squared_error.mean()),"MAE":g.absolute_error.mean(),"P90 absolute error":g.absolute_error.quantile(.90),"P95 absolute error":g.absolute_error.quantile(.95),"P99 absolute error":g.absolute_error.quantile(.99),"Maximum error":g.absolute_error.max(),"Mean residual":g.residual.mean(),"Large-error fraction":(g.absolute_error>P95).mean(),"Support score mean":g.combined_support_score.mean()})
 return rows
regions={"North":oof.lat>=0,"South":oof.lat<0,"Equatorial":oof.lat.abs()<10,"High latitudes":oof.lat.abs()>=60,"Mid latitudes":oof.lat.abs().between(30,60,inclusive="left"),"Low latitudes":oof.lat.abs()<30,
"Americas block":(oof.lon>=-140)&(oof.lon<-30),"Asia block":(oof.lat>=10)&(oof.lat<60)&(oof.lon>=60)&(oof.lon<150)}
pd.DataFrame(tail_table(regions,"region")).to_csv(ART/"regional_tail_risk.csv",index=False)

oof["gap_bucket"]=pd.cut(oof.gap_months,[0,1,3,6,12,np.inf],include_lowest=True).astype(str); oof["season"]=(oof.month%12//3).map({0:"DJF",1:"MAM",2:"JJA",3:"SON"})
temp_rows=[]
for col in ["fold_id","year","month","gap_bucket","season"]:
 for key,g in oof.groupby(col,observed=True): temp_rows.append({"Group type":col,"Group":key,"Row count":len(g),"RMSE":np.sqrt(g.squared_error.mean()),"MAE":g.absolute_error.mean(),"P95 absolute error":g.absolute_error.quantile(.95),"Maximum error":g.absolute_error.max(),"Mean residual":g.residual.mean(),"Large-error fraction":(g.absolute_error>P95).mean(),"Spatial support average":g.spatial_support_score.mean(),"Temporal support average":g.temporal_support_score.mean()})
pd.DataFrame(temp_rows).to_csv(ART/"temporal_tail_risk.csv",index=False)
reg_rows=[]
for col in ["climate_regime","drought_regime"]:
 for key,g in oof.groupby(col): reg_rows.append({"Regime type":col,"Regime":key,"Row count":len(g),"RMSE":np.sqrt(g.squared_error.mean()),"MAE":g.absolute_error.mean(),"P95 absolute error":g.absolute_error.quantile(.95),"Maximum error":g.absolute_error.max(),"Mean residual":g.residual.mean(),"Large-error fraction":(g.absolute_error>P95).mean(),"Support mean":g.combined_support_score.mean(),"Underrepresented":len(g)<.05*len(oof)})
pd.DataFrame(reg_rows).to_csv(ART/"regime_tail_risk.csv",index=False)

q=oof.target.quantile([0,.05,.25,.75,.95,1]).to_numpy(); oof["target_quantile_group"]=pd.cut(oof.target,q,labels=["0-5%","5-25%","25-75%","75-95%","95-100%"],include_lowest=True,duplicates="drop")
ext=[]
for key,g in oof.groupby("target_quantile_group",observed=True): ext.append({"Target quantile":key,"Row count":len(g),"RMSE":np.sqrt(g.squared_error.mean()),"MAE":g.absolute_error.mean(),"Mean residual":g.residual.mean(),"Underprediction rate":g.underprediction_flag.mean(),"Overprediction rate":g.overprediction_flag.mean(),"Relative absolute error":np.mean(g.absolute_error/(g.target.abs()+.1)),"P95 absolute error":g.absolute_error.quantile(.95),"Shrinkage interpretation":"positive residual implies underprediction; negative implies overprediction"})
pd.DataFrame(ext).to_csv(ART/"extreme_target_performance.csv",index=False)

# Test diagnostic overlays; no test target is used.
test_support=sp_test[["row_id","lat","lon","spatial_support_score","spatial_support_group","seen_location","nearest_training_km","spatial_density"]].merge(tm_test[["row_id","temporal_support_score","temporal_support_group","gap_months"]],on="row_id")
test_support["combined_support_score"]=(test_support.spatial_support_score+test_support.temporal_support_score)/2
test_features=test[["ID","SPEI_03_t","SPEI_12_t"]].rename(columns={"ID":"sample_id"})
test_support=test_support.merge(test_features,left_on="row_id",right_on="sample_id"); test_support["climate_regime"]=pd.cut(test_support.SPEI_12_t,[-np.inf,-1,1,np.inf],labels=["dry","normal","wet"]).astype(str); test_support["drought_regime"]=pd.cut(test_support.SPEI_03_t,[-np.inf,-1.5,-.5,.5,1.5,np.inf],labels=["severe_dry","dry","neutral","wet","very_wet"]).astype(str)
test_delta=best.Target-canon.Target; applied=np.abs(test_delta)>1e-12; test_support["correction_applied"]=applied.to_numpy(); test_support["correction"]=test_delta.to_numpy()
overlay=[]
for key,g in test_support.groupby("spatial_support_group"):
 overlay.append({"Support group":key,"Rows":len(g),"Correction applied rows":g.correction_applied.sum(),"Applied fraction":g.correction_applied.mean(),"Spatial support mean":g.spatial_support_score.mean(),"Temporal support mean":g.temporal_support_score.mean(),"Combined support mean":g.combined_support_score.mean(),"Southern share":(g.lat<0).mean(),"Low-support corrected fraction":g.loc[g.correction_applied,"spatial_support_group"].isin(["Low support","Out-of-support"]).mean(),"Maximum correction":g.correction.abs().max(),"Mean absolute correction":g.correction.abs().mean()})
pd.DataFrame(overlay).to_csv(ART/"correction_support_overlay.csv",index=False)
freq_cl=test_support.climate_regime.value_counts(normalize=True); freq_dr=test_support.drought_regime.value_counts(normalize=True)
flags=pd.DataFrame({"row_id":test_support.row_id,"low_spatial_support":test_support.spatial_support_group.isin(["Low support","Out-of-support"]),"temporal_extrapolation":test_support.temporal_support_group.eq("Temporal extrapolation"),
 "unseen_location":~test_support.seen_location,"rare_climate_regime":test_support.climate_regime.map(freq_cl)<.05,"rare_drought_regime":test_support.drought_regime.map(freq_dr)<.05,"southern_hemisphere":test_support.lat<0,"high_latitude":test_support.lat.abs()>=60,
 "correction_applied":test_support.correction_applied,"large_correction":test_support.correction.abs()>=.05-1e-12})
flagcols=[x for x in flags if x!="row_id"]; flags["risk_flag_count"]=flags[flagcols].sum(1); flags["multiple_risk_flags"]=flags.risk_flag_count>=2; flags.to_csv(ART/"test_row_risk_flags.csv",index=False)

completion=[]
mapping={"Median absolute error":"Median absolute error","75th percentile absolute error":"P75 absolute error","90th percentile absolute error":"P90 absolute error","95th percentile absolute error":"P95 absolute error","99th percentile absolute error":"P99 absolute error","Maximum absolute error":"Maximum absolute error","Large-error row fraction":"Large-error fraction >P95","Residual skewness":"Residual skewness","Residual kurtosis":"Residual kurtosis"}
for old,new in mapping.items(): completion.append({"Metric":old,"Notebook 24 status":"not available","Notebook 25 value":metrics[new],"Data source":"row-level strict temporal OOF","Validation strategy":strategy.iloc[0]["Strategy name"],"Limitation":strategy.iloc[0].Limitations})
pd.DataFrame(completion).to_csv(ART/"notebook24_metric_completion.csv",index=False)

dictionary=[]
descs={"row_id":"original sample identifier","original_index":"Train.csv row index","date":"validation date","target":"observed target","oof_prediction":"strict OOF prediction","residual":"target - prediction","spatial_support_score":"fixed 0-1 spatial support","temporal_support_score":"fixed 0-1 temporal support","combined_support_score":"mean spatial/temporal support"}
for c in cols: dictionary.append({"Column name":c,"Type":str(oof[c].dtype),"Description":descs.get(c,c.replace("_"," ")),"Source":"Train.csv / fold model / deterministic audit","Leakage-safe status":"yes; fold-training data only" if c not in ["target","residual","absolute_error","squared_error"] else "validation outcome; never used as model input","Missing value policy":"no missing expected"})
pd.DataFrame(dictionary).to_csv(ART/"oof_data_dictionary.csv",index=False)
readme=f"""# OOF artefact README

Purpose: persistent row-level errors for tail-risk and support audit. Strategy: **{strategy.iloc[0]['Strategy name']}** with two complete-date strict temporal folds. Residual is always `target - oof_prediction`; positive means underprediction.

The known/masked expert structure and 0.65/0.35 blend are canonical-equivalent, but `prior_global_month` is rebuilt with strict date-grouped history to remove the canonical location-major ordering risk. Therefore this artefact must not be described as exact canonical OOF. Validation targets are outputs only and never feature inputs.

Use it for aggregate error, tail, regional, temporal, regime and support diagnostics. Do not use it to tune a new leaderboard model, infer test accuracy, claim unseen-space robustness (same-grid locations dominate), or compare directly with public RMSE. Cache files are accepted only when the frozen policy key and required fold files match.
"""
(ART/"oof_artifact_readme.md").write_text(readme,encoding="utf-8")

low_test=test_support.spatial_support_group.isin(["Low support","Out-of-support"]).mean(); worst_region=pd.read_csv(ART/"regional_tail_risk.csv").sort_values("P95 absolute error",ascending=False).iloc[0]
worst_time=pd.read_csv(ART/"temporal_tail_risk.csv").query("`Group type` == 'fold_id'").sort_values("P95 absolute error",ascending=False).iloc[0]
overlay_ap=test_support[test_support.correction_applied]
final=f'''# Notebook 25 final audit recommendation

1. **Valid row-level OOF:** Yes—{strategy.iloc[0]["Terminology"]}; strict chronology and unique-row assertions passed.
2. **OOF coverage:** {len(oof)/len(train):.3%} ({len(oof):,}/{len(train):,}).
3. **Key tail metric:** P95 absolute error {P95:.6f}; maximum {oof.absolute_error.max():.6f}.
4. **Largest regional tail:** {worst_region.Group}, P95 {worst_region['P95 absolute error']:.6f}.
5. **Largest fold tail:** {worst_time.Group}, P95 {worst_time['P95 absolute error']:.6f}.
6. **Low-support error:** consult support tables; Spearman(combined support, absolute error)={rho_abs:.4f}, bootstrap CI [{np.quantile(boot,.025):.4f},{np.quantile(boot,.975):.4f}]. Association is diagnostic, not causal.
7. **Stronger support relationship:** compare spatial/temporal group tables; same-grid spatial support has limited variation, so temporal evidence is more informative here.
8. **Test low/out spatial support:** {low_test:.3%}; no target-based performance claim is made.
9. **Corrected test support:** {len(overlay_ap):,} rows; mean combined support {overlay_ap.combined_support_score.mean():.4f}; low/out spatial share {overlay_ap.spatial_support_group.isin(['Low support','Out-of-support']).mean():.3%}.
10. **Risk flags:** diagnostic only; see `test_row_risk_flags.csv`.
11. **Notebook 24 completion:** {len(completion)} previously unavailable tail metrics now populated from proxy OOF.
12. **Model card:** update with proxy-OOF tail numbers, same-grid spatial-support limitation, and test temporal-extrapolation profile.
13. **Preserve current-best:** Yes; submission hashes remain unchanged.
14. **New score optimization:** No.
15. **Report readiness:** Yes, with the canonical-equivalent/proxy terminology and limitations retained.

**Final decisions:** OOF artefact valid; tail/support audit complete; current-best retained; score optimization closed. Notebook 26 should be `26_final_competition_report_and_repository_consolidation.ipynb`, a documentation/consolidation step only.
'''
(ART/"final_audit_recommendation.md").write_text(final,encoding="utf-8"); display(Markdown(final))
required=["input_integrity_report.csv","oof_strategy_definition.csv","oof_row_level_predictions.parquet","oof_row_level_predictions.csv.gz","oof_quality_report.csv","global_tail_risk_metrics.csv","worst_error_rows.csv","spatial_support_rows.csv","spatial_support_performance.csv","test_spatial_support_audit.csv","temporal_support_rows.csv","temporal_support_performance.csv","combined_support_error_relationship.csv","regional_tail_risk.csv","temporal_tail_risk.csv","regime_tail_risk.csv","extreme_target_performance.csv","correction_support_overlay.csv","test_row_risk_flags.csv","notebook24_metric_completion.csv","oof_data_dictionary.csv","oof_artifact_readme.md","final_audit_recommendation.md"]
assert all((ART/x).exists() for x in required); assert sha(CANON)==CANON_SHA and sha(BEST)==BEST_SHA
stamp("complete")


[strict_temporal_1] elapsed=10.1s


[strict_temporal_2] elapsed=12.9s


[row-level OOF and support] elapsed=76.7s


# Notebook 25 final audit recommendation

1. **Valid row-level OOF:** Yes—canonical-equivalent OOF, not exact canonical OOF; strict chronology and unique-row assertions passed.
2. **OOF coverage:** 34.774% (749,046/2,154,021).
3. **Key tail metric:** P95 absolute error 1.520082; maximum 4.461003.
4. **Largest regional tail:** South, P95 1.773752.
5. **Largest fold tail:** strict_temporal_2, P95 1.556778.
6. **Low-support error:** consult support tables; Spearman(combined support, absolute error)=-0.1956, bootstrap CI [-0.2097,-0.1824]. Association is diagnostic, not causal.
7. **Stronger support relationship:** compare spatial/temporal group tables; same-grid spatial support has limited variation, so temporal evidence is more informative here.
8. **Test low/out spatial support:** 0.000%; no target-based performance claim is made.
9. **Corrected test support:** 36,832 rows; mean combined support 0.7572; low/out spatial share 0.000%.
10. **Risk flags:** diagnostic only; see `test_row_risk_flags.csv`.
11. **Notebook 24 completion:** 9 previously unavailable tail metrics now populated from proxy OOF.
12. **Model card:** update with proxy-OOF tail numbers, same-grid spatial-support limitation, and test temporal-extrapolation profile.
13. **Preserve current-best:** Yes; submission hashes remain unchanged.
14. **New score optimization:** No.
15. **Report readiness:** Yes, with the canonical-equivalent/proxy terminology and limitations retained.

**Final decisions:** OOF artefact valid; tail/support audit complete; current-best retained; score optimization closed. Notebook 26 should be `26_final_competition_report_and_repository_consolidation.ipynb`, a documentation/consolidation step only.


[complete] elapsed=88.3s
